## DATA INGESTION


In [1]:
### Document Structure

from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is main text content I am using to create RAG",
    metadata={
        'source':'bliss_corpus.json',
        'pages':1,
        'author':'Subodh',
        'date_created':'2026-09-22'
    }
)

print(doc)

page_content='This is main text content I am using to create RAG' metadata={'source': 'bliss_corpus.json', 'pages': 1, 'author': 'Subodh', 'date_created': '2026-09-22'}


In [3]:
### create a simple txt file
import os
os.makedirs('../data/text_files', exist_ok=True)

In [4]:
# LOADING JSON FILES

from langchain_community.document_loaders import JSONLoader
from pathlib import Path

file_path = Path("../data/json_files/bliss_corpus.json")

def extract_metadata(record: dict, metadata: dict) -> dict:
    metadata["doc_id"] = record.get("doc_id")
    metadata["source"] = record.get("metadata", {}).get("source")
    metadata["url"] = record.get("metadata", {}).get("url")
    metadata["topic_group"] = record.get("metadata", {}).get("topic_group")
    metadata["flags"] = record.get("metadata", {}).get("flags")
    return metadata

## JSON loader with jq schema and content key
loader = JSONLoader(
    file_path=str(file_path),
    jq_schema=".[]",  
    content_key='. | "Question: " + .question + "\nAnswer: " + .answer',
    is_content_key_jq_parsable=True,  
    metadata_func=extract_metadata,
)

documents = loader.load()
documents[:5]

C:\Users\my pc\AppData\Local\Temp\ipykernel_8412\909986912.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader
d:\RAG_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'NIMH', 'seq_num': 1, 'doc_id': 'nimh_5-action-steps-to-help-someone-having-thoughts-of-suicide_01', 'url': 'https://www.nimh.nih.gov/health/publications/5-action-steps-to-help-someone-having-thoughts-of-suicide', 'topic_group': 'suicide_selfharm_crisis', 'flags': ['safety_sensitive']}, page_content='Question: If I\'m worried someone might be suicidal, is it okay to just ask them directly?\nAnswer: Yes. Directly asking someone "Are you thinking about suicide?" is the first of the 5 action steps. Research shows that asking people if they are suicidal does not increase suicidal behavior or thoughts, and the question can help open up a conversation.'),
 Document(metadata={'source': 'NIMH', 'seq_num': 2, 'doc_id': 'nimh_5-action-steps-to-help-someone-having-thoughts-of-suicide_02', 'url': 'https://www.nimh.nih.gov/health/publications/5-action-steps-to-help-someone-having-thoughts-of-suicide', 'topic_group': 'suicide_selfharm_crisis', 'flags': ['safety_sensitiv

### Creating Data Chunks

In [5]:
### Not Performing Chunking on Json Data because:
# each document in json dataset represents a self-contained Question & Answer pair that is already short and focused
# In RAG pipelines, chunking is designed to break down large documents (like PDFs or long articles) into 
# smaller, semantically coherent pieces so that search queries match relevant paragraphs instead of huge walls of text.

### Embeddings and VectorStore DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [7]:
## BAAI/bge-small-en-v1.5 for Json dataset
from typing import List
import numpy as np
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer

class EmbeddedDocument:
    """"Pairs an Embedding Vector with it's source Document."""
    embedding: np.ndarray
    text: str
    metadata: Dict[str, Any]

class EmbeddingManager:
    """Handles document embeddings generation using SentenceTransformer and manages storage in Vector Database."""

    def __init__(self, model_name: str = "BAAI/bge-small-en-v1.5"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated. Shape: {embeddings.shape}")
        return embeddings


    def embed_documents(self, documents: List[Document]) -> List[EmbeddedDocument]:
        """
        Extracts page_content from LangChain Document objects and generates embeddings.
        One Vector per Document, preserving the original text and metadata.
        Args:
            documents: List of LangChain Document objects.
        Returns:
            List of EmbbeddedDocument, each holding the embedding vector, text, and metadata from the original Document.
        """
        if not documents:
            raise ValueError("No documents provided for embedding.")

        texts = [doc.page_content for doc in documents]
        embeddings = self.generate_embeddings(texts)
        return[
            EmbeddedDocument(
                embedding=embeddings[i],
                text=documents[i].page_content,
                metadata=documents[i].metadata
            )
            for i in range(len(documents))
        ]

## initialize the embedding manager for JSON dataset
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: BAAI/bge-small-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1896.47it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStrore

In [8]:
## Vector Store for JSON dataset
class VectorStore:
    """Manages document embeddings in a ChromaDB Vector Store"""

    def __init__(self,collection_name: str = 'json_qa_documents', persist_directory: str = '../data/vector_store'):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "JSON Q&A document embeddings for RAG",
                    "hnsw:space": "cosine"
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def _flatten_list(self, metadata: Dict[str, Any]) -> Dict[str, Any]:
        """
        Flatten nested lists in metadata for ChromaDB compatibility.
        ChromaDB only accepts str/int/float/bool metadata values.
        JSON docs carry list fields ('flags': ['safety_sensitive', ...]),
        so flatten any list into a comma-separated string. 
        """
        clean = {}
        for k, v in metadata.items():
            if isinstance(v, list):
                clean[k] = ", ".join(str(x) for x in v)
            elif v is None:
                clean[k] = ""
            else:
                clean[k] = v
        return clean

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = doc.metadata.get("doc_id") or f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = self._flatten_list(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: json_qa_documents
Existing documents in collection: 2964


In [9]:
### Convert Text to embeddings and store in Vector Store
json_texts = [doc.page_content for doc in documents]

### Generate embeddings for the text documents
embeddings = embedding_manager.generate_embeddings(json_texts)

### Add documents and embeddings to the vector store
vectorstore.add_documents(documents, embeddings)

Generating embeddings for 2964 texts...


Batches: 100%|██████████| 93/93 [04:13<00:00,  2.73s/it]


Embeddings generated. Shape: (2964, 384)
Adding 2964 documents to vector store...
Successfully added 2964 documents to vector store
Total documents in collection: 2964


### Retriever Pipeline From VectorStore

## Hybrid Retriever Pipeline From VectorStore

**Hybrid Retriever** : `Dense + BM25 via Reciprocal Rank Fusion`

Evaluation against `retriever_evaluation.ipynb` / `hybrid_search_evaluation.ipynb` compared the pure-dense `RAGRetriever` against a BM25-only retriever and a hybrid combination fused with **Reciprocal Rank Fusion (RRF)**. Results were close (hybrid: +1.3pp Hit@1 and +0.9pp MRR vs. dense; -2.0pp Hit@5), but the hybrid retriever is added here as the production retriever.

The BM25 index is built once from the same documents already stored in the `VectorStore`'s ChromaDB collection, so it can never drift out of sync with the dense index. `HybridRetriever` keeps the exact same `retrieve()` interface as `RAGRetriever` (same input args, same output shape) so it's a drop-in replacement anywhere `rag_retriever` was used, including `rag_simple()` below.

In [12]:
import re
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> List[str]:
    """Simple, dependency-free tokenizer: lowercase, strip punctuation, split on whitespace."""
    return re.findall(r"[a-z0-9]+", text.lower())


class BM25Index:
    """Builds and queries a BM25 index over the documents already stored in a VectorStore's collection."""

    def __init__(self, vector_store: VectorStore):
        all_docs = vector_store.collection.get(include=["documents"])
        self.doc_ids = all_docs["ids"]
        self.doc_texts = all_docs["documents"]
        tokenized_corpus = [tokenize(doc) for doc in self.doc_texts]
        self.bm25 = BM25Okapi(tokenized_corpus)
        print(f"BM25 index built over {len(self.doc_ids)} documents")

    def query(self, query_text: str, top_k: int) -> List[Dict[str, Any]]:
        """Returns top_k candidates ranked best-first as {'id', 'content', 'score'} dicts."""
        scores = self.bm25.get_scores(tokenize(query_text))
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [
            {"id": self.doc_ids[i], "content": self.doc_texts[i], "score": float(scores[i])}
            for i in top_indices
        ]


bm25_index = BM25Index(vectorstore)


BM25 index built over 2964 documents


In [13]:
class HybridRetriever:
    """Combines dense (cosine-similarity) retrieval with BM25 lexical retrieval via Reciprocal Rank Fusion.

    Keeps the same retrieve() interface as RAGRetriever: same arguments, same output shape
    (list of dicts with 'id', 'content', 'metadata', 'similarity_score', 'distance', 'rank'),
    so it's a drop-in replacement wherever a retriever is used.
    """

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager,
        bm25_index: BM25Index,
        fusion_pool_size: int = 20,
        rrf_k: int = 10,
    ):
        """
        Initialize the Hybrid Retriever

        Args:
            vector_store: Vector Store containing document embeddings (dense side)
            embedding_manager: Manager for generating query embeddings (dense side)
            bm25_index: Prebuilt BM25 index over the same corpus (lexical side)
            fusion_pool_size: Number of candidates pulled from EACH retriever before fusion
            rrf_k: RRF damping constant (60 is the standard default)
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        self.bm25_index = bm25_index
        self.fusion_pool_size = fusion_pool_size
        self.rrf_k = rrf_k

    def _dense_candidates(self, query: str, pool_size: int, where: Dict[str, Any] = None) -> Dict[str, Dict[str, Any]]:
        """Returns {doc_id: {'content', 'metadata', 'similarity_score', 'distance'}} from the dense retriever."""
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        query_kwargs = {"query_embeddings": [query_embedding.tolist()], "n_results": pool_size}
        if where:
            query_kwargs["where"] = where
        results = self.vector_store.collection.query(**query_kwargs)

        candidates = {}
        if results['documents'] and results['documents'][0]:
            for doc_id, document, metadata, distance in zip(
                results['ids'][0], results['documents'][0], results['metadatas'][0], results['distances'][0]
            ):
                candidates[doc_id] = {
                    "content": document,
                    "metadata": metadata,
                    "similarity_score": 1 - distance,
                    "distance": distance,
                }
        return candidates

    def retrieve(
        self, query: str, top_k: int = 3, score_threshold: float = 0.0, where: Dict[str, Any] = None
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query using fused dense + BM25 ranking.

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum fused RRF score (RRF scores are small and NOT comparable
                to the dense retriever's cosine-similarity scores — leave at 0.0 unless you've
                separately calibrated a threshold for this fused scale)
            where: Additional filtering criteria for the query (applied on the dense side only,
                since ChromaDB metadata filtering has no BM25 equivalent here)
        Returns:
            List of dictionaries containing retrieved documents and metadata, same shape as RAGRetriever.retrieve()
        """
        print(f"Retrieving documents for query: '{query}' (hybrid: dense + BM25)")
        print(f"Top K: {top_k}, Fusion pool size: {self.fusion_pool_size}, RRF k: {self.rrf_k}")

        try:
            dense_candidates = self._dense_candidates(query, self.fusion_pool_size, where=where)
            bm25_candidates = self.bm25_index.query(query, self.fusion_pool_size)

            rrf_scores: Dict[str, float] = {}
            for rank, doc_id in enumerate(dense_candidates.keys(), start=1):
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (self.rrf_k + rank)
            for rank, cand in enumerate(bm25_candidates, start=1):
                rrf_scores[cand["id"]] = rrf_scores.get(cand["id"], 0.0) + 1.0 / (self.rrf_k + rank)

            fused_order = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

            bm25_lookup = {c["id"]: c for c in bm25_candidates}

            retrieved_docs = []
            for i, (doc_id, fused_score) in enumerate(fused_order):
                if fused_score < score_threshold:
                    continue

                if doc_id in dense_candidates:
                    content = dense_candidates[doc_id]["content"]
                    metadata = dense_candidates[doc_id]["metadata"]
                    distance = dense_candidates[doc_id]["distance"]
                else:
                    content = bm25_lookup[doc_id]["content"]
                    metadata = {}
                    distance = None

                retrieved_docs.append({
                    'id': doc_id,
                    'content': content,
                    'metadata': metadata,
                    'similarity_score': fused_score,   # fused RRF score, not a cosine similarity
                    'distance': distance,
                    'rank': i + 1,
                })

                if len(retrieved_docs) >= top_k:
                    break

            print(f"Retrieved {len(retrieved_docs)} documents (after fusion)")
            return retrieved_docs

        except Exception as e:
            print(f"Error during hybrid retrieval: {e}")
            return []


hybrid_retriever = HybridRetriever(vectorstore, embedding_manager, bm25_index)
hybrid_retriever


In [14]:
hybrid_retriever.retrieve(query="How should I keep someone safe from suicide?",)


Retrieving documents for query: 'How should I keep someone safe from suicide?' (hybrid: dense + BM25)
Top K: 3, Fusion pool size: 20, RRF k: 10
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.64it/s]

Embeddings generated. Shape: (1, 384)
Retrieved 3 documents (after fusion)


[{'id': 'nimh_5-action-steps-to-help-someone-having-thoughts-of-suicide_03',
  'content': "Question: How can I help keep someone safe if they're having suicidal thoughts?\nAnswer: You can help by reducing their access to highly lethal items or places. Ask the person if they have a specific plan, and work to make any lethal means less available or less dangerous, since this can help keep them safe when suicidal thoughts arise.",
  'metadata': {'content_length': 337,
   'source': 'NIMH',
   'url': 'https://www.nimh.nih.gov/health/publications/5-action-steps-to-help-someone-having-thoughts-of-suicide',
   'topic_group': 'suicide_selfharm_crisis',
   'flags': 'safety_sensitive',
   'doc_id': 'nimh_5-action-steps-to-help-someone-having-thoughts-of-suicide_03',
   'seq_num': 3,
   'doc_index': 2},
  'similarity_score': 0.17424242424242425,
  'distance': 0.1423783302307129,
  'rank': 1},
 {'id': 'helios_59',
  'content': "Question: How do I stop suicidal thoughts?\nAnswer: Keep in mind that t

### Integration VectorDB Context pipeline with LLM output

In [15]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initialize the Groq LLM (set your Groq_API_Key in .env)
groq_api_key = os.getenv('GROQ_API_KEY')
llm = ChatGroq(groq_api_key=groq_api_key, model_name='qwen/qwen3.8-27b', temperature=0.1,max_tokens=1024)

##simple RAG funtion: retrieve context + generate responses
def rag_simple(query,retriever,llm,top_k=3):

    ##retrieve the context
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using Groq LLM
    prompt = f"""Use the following context to answer the question concisely.
                Context:
                {context}

                Question: {query}

                Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [17]:
# Same RAG pipeline, now backed by the hybrid retriever instead of the pure-dense retriever
answer_hybrid = rag_simple("What is Mental Health?", hybrid_retriever, llm)
print(answer_hybrid)


Retrieving documents for query: 'What is Mental Health?' (hybrid: dense + BM25)
Top K: 3, Fusion pool size: 20, RRF k: 10
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.48it/s]

Embeddings generated. Shape: (1, 384)
Retrieved 3 documents (after fusion)


Mental health is our mental well-being, which includes our beliefs, thoughts, feelings, and behaviors. It encompasses our emotions, our ability to solve problems and overcome difficulties, our social connections, and our understanding of the world around us.
